# This code is meant to replace the combination of LAHM_analysis & LAHM_analysis_analytical

April 2021

In [15]:
# Generic resources
import numpy as np
import matplotlib.pyplot as plt
from importlib import reload
from scipy import optimize

# Local imports
import sys; sys.path.append('../')
import LAHM_library as LL

In [17]:
%matplotlib inline

In [19]:
# This is the exponential-with-time function
def test_func(t, A, t0): #first varible is the independent varible, then the subsequent are the ...
    return A*(1-np.exp(-t/t0))

In [21]:
# Instrument factors
myLAHM_Parameters = LL.LAHM_Parameters(inst_number = 10)
myLAHM_Parameters.report()

Parameters:
LAHM unit = 10
inst_factor = 0.77
exponential_prefactor = 3.43
temperature_factor = 0.238
ugbasic_offset = 3.0


In [23]:
# Graphics parameters
figwidth = 4
figheight = 3
fontsize = 16

In [25]:
# Name of the master file with all the filenames and the volumes
# filename = 'filelist_figure5.txt'
# filename = 'filelist_algae.txt'
filename = 'filelist.txt'

In [27]:
# Read in the master file
line0_list, number = LL.getline0list(filename)
for i in range(number):
    print(line0_list[i])

filelist.txt
../LAHM Data Summer 2025/LAHM variable tests/run-1.txt 600
../LAHM Data Summer 2025/LAHM variable tests/UPS2020_21L1.txt 600
../LAHM Data Summer 2025/LAHM variable tests/UPS2020_21L2.txt 600
../LAHM Data Summer 2025/LAHM variable tests/UPS2020_21F1.txt 600


In [29]:
# Run the original Schmitt analysis
myTtrace_list = LL.getgpg(line0_list, myLAHM_Parameters)
for i in range(0,number):
    myTtrace = myTtrace_list[i]
    myTtrace.report()

From getgpg: number =  4
From getgpg: t =  0


ValueError: could not convert string to float: 'Data'

In [31]:
pwd

'/Users/emma/Dropbox/2025 LAHM/Trial runs'

In [ ]:
# Plot all the traces
plt.figure(figsize=(figwidth, figheight))
mylabel_List = []
for i in range(number): 
    myTtrace = myTtrace_list[i]
   # mylabel = myTtrace.filename+', '+str(myTtrace.ugbasic)[0:4]+'$\ \mu g$'+', '+str(myTtrace.volume)[0:4]+' mL'
    mylabel = myTtrace.filename
    print(mylabel)
    mylabel_List.append(mylabel)
    plt.plot(myTtrace.time, myTtrace.Temp_av, label=mylabel) 
#plt.ylabel('Mean temp increase ($^\circ$C)')
plt.ylabel('Mean temp increase')
plt.xlabel('time (s)')
plt.legend()
plt.grid(True)
plt.show()
# Save the figure
# plt.savefig(filename+'.png', dpi=200)

# Print Label array
# print(mylabel_List)

In [ ]:
## Loops through file list and graphs

#True = want to see every graph
graphs = True

A_array = np.zeros(number)
t0_array = np.zeros(number)
theorylabel_List = []

if graphs:
    plt.figure(figsize=(figwidth, figheight))
    plt.show()

for i in range(number):
    myTtrace = myTtrace_list[i]
    myTtrace.report()
    
    # Extract from when it starts to rise
    Temp_expt = myTtrace.Temp_av[40:160]
    time_expt = myTtrace.time[40:160]-myTtrace.time[40]
    params, params_covariance = optimize.curve_fit(test_func, time_expt, Temp_expt, p0=[2, 3])
    A_array[i]=params[0]
    t0_array[i]=params[1]
   
    if graphs:
        theory2 = test_func(time_expt,params[0],params[1])
        exptlabel = mylabel_List[i]

        #graph
        plt.plot(time_expt,Temp_expt,'+',label=exptlabel)
        theorylabel = '$A(1-e^{-t/t_0})$'+'  A='+str(params[0])[0:6]+'  t0='+str(params[1])[0:6]
        print(theorylabel)
        theorylabel_List.append(theorylabel)
        plt.plot(time_expt,theory2, color= 'black')
        #plt.ylabel('$\Delta T$ ($^\circ$C)')
        plt.ylabel('Delta T')
        plt.xlabel('time (s)')
        plt.legend()
        plt.grid(True)

        #save figure
        #plt.savefig('bestfit.png', dpi=200)
        #plt.savefig(myTtrace.filename+'Fit.png',dpi=200)

print("A = ", A_array)
print("t0 = ",t0_array)

In [ ]:
# # writes data into output.txt file
# f2 = open('output.txt', 'w')
# #header
# f2.write('file         \t \t A \t   t0\n')

# # write the data as float values with a tab between them
# for i in range(number):
#     f2.write('%s    %f    %f\n' % (myTtrace_list[i].filename, A_array[i],t0_array[i]))

# f2.close()

In [ ]:
# Copy micrograms into a new array "microg" from myTtrace_List
microg = np.zeros(number)
for i in range(number):
    print(myTtrace_list[i].ugbasic)
    microg[i] = myTtrace_list[i].ugbasic

# Print in the order they come
print("microg before sorting = ",microg)

# Find the indices of the microgram array in ascending order
sort_index = np.argsort(microg); print('Microgram indices in increasing order =', sort_index)

# Make new arrays of Schmitt's micrograms and A-values, in ascending order of micrograms
microg_sorted = microg[sort_index]; print("microg after sorting = ",microg_sorted)
A_array_sorted = A_array[sort_index]; print("A-values after sorting = ",A_array_sorted)

In [ ]:
# Examining correlation between A and Schmitt's micrograms

if number > 2: 
    
    # Plot A values as a function of Schmitt's values
    plt.figure(figsize=(figwidth, figheight))
    plt.plot(microg_sorted, A_array_sorted,'o', color= 'magenta', label='obs')
    plt.grid(True)
    
    # Do a fit
    maxorder = 2
    if number > 2:
        fit_order = np.min((maxorder,number)); print('fit order = ',fit_order)
        p = np.polyfit(microg_sorted, A_array_sorted,fit_order); print('p = ',p)
        microg_dense = np.linspace(np.min(microg),np.max(microg))
        A_polyval = np.polyval(p,microg_dense)
        plt.plot(microg_dense,A_polyval,'--',label='fit')
#         plt.xlabel('Loading ($\mu$g) (Schmitt et al)') 
        # plt.ylabel('Asymptotic $\Delta$T (assuming exponential form)')
    plt.legend()
    
    print('t0 ranges from ', np.min(t0_array),np.max(t0_array))